#### Simple LLM Application with LCEL (Lang chain expression language)

In this quickstart we'll show you how to build a simple LLM application with LangChain. This application will translate text from English into another language. This is a relatively simple LLM application - it's just a single LLM call plus some prompting. Still, this is a great way to get started with LangChain - a lot of features can be built with just some prompting and an LLM call!

After seeing this video, you'll have a high level overview of:

- Using language models

- Using PromptTemplates and OutputParsers

- Using LangChain Expression Language (LCEL) to chain components together

- Debugging and tracing your application using LangSmith

- Deploying your application with LangServe

In [1]:
import os 
from dotenv import load_dotenv

load_dotenv()

import openai
openai.api_key = os.getenv("OPENAI_API_KEY")

groq_api_key = os.getenv("GROQ_API_KEY")



In [8]:
from langchain_groq import ChatGroq

model=ChatGroq(model="openai/gpt-oss-120b",groq_api_key=groq_api_key)
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.3', 'langchain': '1.4.0'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000021288EB1700>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000021288EB38C0>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [19]:
from langchain_core.messages import HumanMessage, SystemMessage
messages = [
    SystemMessage(content="You are a helpful assistant that translates English to French."),
    HumanMessage(content="Hello How are you?"),
]  

In [20]:
result=model.invoke(messages)

In [21]:
result

AIMessage(content='Bonjour, comment ça va\u202f?', additional_kwargs={'reasoning_content': 'We need to translate English to French. The user says "Hello How are you?" So translation: "Bonjour, comment ça va ?" Possibly punctuation. Provide translation.'}, response_metadata={'token_usage': {'completion_tokens': 50, 'prompt_tokens': 90, 'total_tokens': 140, 'completion_time': 0.103861075, 'completion_tokens_details': {'reasoning_tokens': 34}, 'prompt_time': 0.032516507, 'prompt_tokens_details': None, 'queue_time': 0.411855317, 'total_time': 0.136377582}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_e5b4e54fbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0dcfc-0d44-7193-995d-851c483e10e2-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 90, 'output_tokens': 50, 'total_tokens': 140, 'output_token_details': {'reasoning': 34}})

In [22]:
from langchain_core.output_parsers import StrOutputParser
parser = StrOutputParser()
parsed_msg = parser.invoke(result)

In [23]:
parsed_msg

'Bonjour, comment ça va\u202f?'

In [24]:
chain = model | parser
chain.invoke(messages)

'Bonjour, comment ça va\u202f?'

In [32]:
from langchain_core.prompts import ChatPromptTemplate

generic_template = "Translate the following in to {language}:"

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", generic_template),
        ("user", "{text}")
    ]
)
prompt


ChatPromptTemplate(input_variables=['language', 'text'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['language'], input_types={}, partial_variables={}, template='Translate the following in to {language}:'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['text'], input_types={}, partial_variables={}, template='{text}'), additional_kwargs={})])

In [33]:
result=prompt.invoke({"language":"French","text":"Hello"})

In [34]:
result.to_messages()

[SystemMessage(content='Translate the following in to French:', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Hello', additional_kwargs={}, response_metadata={})]

In [35]:
chain = prompt| model| parser
chain.invoke({"language":"French","text":"Hello"})

'Bonjour'